In [2]:
%%time

# 실행 추적 평가 영역과 ZKSTARK 평가 영역
# T = 실행 추적 평가 영역
# L = ZKSTARK 평가 영역
# T와 L은 서로소
# |T| < |L|
# |T|는 2의 거듭제곱(FFT를 사용하기 위해)
# |L|은 2의 거듭제곱(FFT를 사용하기 위해)
# F[P] = 크기가 P인 소체, P는 소수
# F[P, *] = F[P]의 곱셈 부분군 = {1, 2, 3, ..., P - 1}
# |F[P, *]| = P - 1
# T와 L은 F[P, *]의 부분군 -> |T|와 |L|은 |F[P, *]| = P - 1을 나눔
import fft_poly
import iop
import merkle
import numpy as np
import polynomial
import stark
from fft import eval_poly, fft, ifft
from field import F, find_generator, generate, get_primitive_root
from polynomial import Polynomial, X

from utils import is_pow2, is_prime, min_pow2_gt

### 주장 예시 = 실행 추적의 모든 값이 0 또는 1 ###
trace = [int(i) for i in np.random.choice([0, 1], size=1024)]
trace_len = len(trace)

### 평가 영역 설정 - 소체와 생성원 선택 ###
# 소체 F[P]
# Starkware 소수
# P = 2**251 + 17 * 2**192 + 1
P = 3 * 2**30 + 1
assert is_prime(P)

# 소체 F[P]의 곱셈군 F[P, *]의 생성원
# 증명자와 검증자는 g를 알고 있어야 함
g = find_generator(P)
print("Generator of F[P, *] = g =", g)

### 실행 추적 평가 영역 ###
# T = 실행 추적 평가 영역으로 설정
# FFT를 사용하려면 |T|가 2의 거듭제곱이어야 함
# |T|가 F[P, *]의 부분군이 되려면 |T|가 P - 1을 나누어야 함
assert is_pow2(trace_len)
assert (P - 1) % trace_len == 0
trace_domain = generate(get_primitive_root(g, trace_len, P), trace_len, P)

### 실행 추적 다항식 ###
# t(x) = 실행 추적 다항식
# t(T[i]) = trace[i]
t = fft_poly.interp(trace, trace_domain, P)
assert fft_poly.eval(t, trace_domain, P) == trace, "t(T) != trace"
print("Degree of trace polynomial =", t.degree())

### 제약 다항식 ###
# c(x) = 제약 다항식
# trace[i]는 0 또는 1 -> 모든 T[i]에서 trace[i](trace[i] - 1) = 0
#                           -> 모든 T[i]에서 t(x)(t(x) - 1) = 0
#                           -> c(x) = t(x)(t(x) - 1)로 설정
c = t * t - t
print("Degree of constraint polynomial =", c.degree())

EXP_FACTOR = 8
N = EXP_FACTOR * trace_len

prover = stark.Prover(
    P=P,
    g=g,
    trace_poly=t,
    trace_domain=trace_domain,
    exp_factor=EXP_FACTOR,
    constraint_poly=c
)

x = X(1, lambda x: F(x, P))

verifier = stark.Verifier(
    P=P,
    g=g,
    trace_len=trace_len,
    exp_factor=EXP_FACTOR,
    constraint_poly=x * x - x
)

### 통신 채널 ###
chan = iop.Channel(iop.Prover(prover), iop.Verifier(verifier))

### STARK와 FRI - 증명자의 커밋 ###
prover.commit(chan)

### STARK - 검증자의 질의 ###
# 질의 전 사전 검사
verifier.check()

NUM_QUERIES = 50
assert NUM_QUERIES <= N

Q = [int(i) for i in np.random.choice(range(N), size=NUM_QUERIES, replace=False)]
for i in Q:
    verifier.query(i, chan)

### FRI - 검증자의 질의 ###
for i in Q:
    verifier.fri().query(i, chan)

print("All checks passed ✅️")

# TODO - 공격
# TODO - 확률?
# TODO: 머클 트리에서 t(x)와 t(-x)를 쌍으로 묶을까?

Generator of F[P, *] = g = 5
Degree of trace polynomial = 1023
Degree of constraint polynomial = 2046
All checks passed ✅️
CPU times: user 1.54 s, sys: 2.03 ms, total: 1.54 s
Wall time: 1.54 s


In [1]:
# 간단한 zk-STARK 증명의 단계별 설명

# N = 4
# A = [1, 0, 1, 1]
# g = 4 mod 17
# i < N에서 f(g[i]) = A[i]
# A[i] = 0 또는 1 -> i < N에서 A[i](A[i] - 1) = 0 -> f(g[i])^2 - f(g[i]) = 0
# c(x) = f(x)^2 - f(x)
# u(x) = ((x-g[0])(x - g[1])...(x - g[N - 1])) = x^N - 1
# p(x) = c(x) / u(x)
# p(x)는 차수 <= deg(c) - N인 다항식 <-> x = g[i]에서 c(x) = 0 <-> 모든 A[i] = 0 또는 1

# 질의
# 검증자가 p(x)u(x) = c(x)를 확인(L - G에 속하는 x를 질의, G에 속하는 x를 질의하면 영지식성이 깨짐)
# 1. 검증자가 L - G에 속하는 z를 질의
# 2. 증명자가 p(z), f(z)와 각각의 머클 증명을 전송
# 3. 검증자가 p(z)u(z) = c(z)와 머클 증명을 확인

# 부정행위를 높은 확률로 탐지하는 데 필요한 질의 수
# 다항식 차수 = d = 5
# 평가 영역 L = 100
# 거짓 평가값 수 = |L| - d = 95
# k번 질의 후 적발될 확률 = 1 - k번 질의 후 적발되지 않을 확률 ~ 1 - (d / |L|)^k

# TODO - p가 저차수가 아닐 때의 부정행위
# TODO - 거짓 실행 추적의 예시?

# p에 FRI 적용

Generator of F[P, *] = 5
Generator of G = 1855261384
Generator of L = 1734477367
